In [1]:
# ============================================
# CELDA 1: IMPORTAR LIBRERÍAS
# ============================================

import os
import pandas as pd
import numpy as np

from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE

import joblib

In [2]:
# ============================================
# CELDA 2: RUTAS DEL DATASET
# ============================================

INPUT_DATASET = r"e:\Proyectos VS code\ProcesamientoData\output_phase3\transformed_it_ot.csv"

OUTPUT_DIR = r"e:\Proyectos VS code\ProcesamientoData\output_phase4"

OUTPUT_BALANCED = os.path.join(
    OUTPUT_DIR,
    "balanced_it_ot.csv"
)

OUTPUT_PCA = os.path.join(
    OUTPUT_DIR,
    "reduced_it_ot.csv"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("INPUT:")
print(INPUT_DATASET)

print("\nOUTPUT BALANCED:")
print(OUTPUT_BALANCED)

print("\nOUTPUT PCA:")
print(OUTPUT_PCA)

INPUT:
e:\Proyectos VS code\ProcesamientoData\output_phase3\transformed_it_ot.csv

OUTPUT BALANCED:
e:\Proyectos VS code\ProcesamientoData\output_phase4\balanced_it_ot.csv

OUTPUT PCA:
e:\Proyectos VS code\ProcesamientoData\output_phase4\reduced_it_ot.csv


In [3]:
# ============================================
# CELDA 3: VALIDAR DATASET
# ============================================

if os.path.exists(INPUT_DATASET):
    print("DATASET ENCONTRADO")
else:
    print("DATASET NO EXISTE")

DATASET ENCONTRADO


In [4]:
# ============================================
# CELDA 4: CARGAR DATASET
# ============================================

df = pd.read_csv(INPUT_DATASET)

print(df.head())

print("\nSHAPE:")
print(df.shape)

print("\nCOLUMNAS:")
print(df.columns)

   src_port  dst_port  protocol  duration_ms  bidirectional_packets  \
0 -2.740926 -0.353949 -1.153072    -0.159071              -0.006939   
1  0.833272 -0.209669  1.697311    -0.157960              -0.005903   
2 -2.740926 -0.353949 -1.153072    -0.159071              -0.006939   
3 -1.362775 -0.349924  1.697311    -0.158824              -0.006766   
4 -2.740926 -0.353949 -1.153072    -0.159071              -0.006939   

   bidirectional_bytes  src2dst_packets  dst2src_packets  src2dst_bytes  \
0            -0.016591        -0.004337        -0.006445      -0.010569   
1            -0.011866        -0.003011        -0.006445      -0.004041   
2            -0.016591        -0.004337        -0.006445      -0.010569   
3            -0.015823        -0.004337        -0.006132      -0.010413   
4            -0.016591        -0.004337        -0.006445      -0.010569   

   dst2src_bytes  ...  application_name  application_category  attack_type  \
0      -0.018081  ...               100     

In [5]:
# ============================================
# CELDA 5: SEPARAR FEATURES Y TARGET
# ============================================

X = df.drop(columns=["traffic_label"])

y = df["traffic_label"]

print("FEATURES:")
print(X.shape)

print("\nTARGET:")
print(y.shape)

print("\nDISTRIBUCIÓN ORIGINAL:")
print(y.value_counts())

FEATURES:
(3907536, 21)

TARGET:
(3907536,)

DISTRIBUCIÓN ORIGINAL:
traffic_label
0    3822077
1      85459
Name: count, dtype: int64


In [6]:
# ============================================
# CELDA 6: BALANCEO CON SMOTE
# ============================================

smote = SMOTE(
    sampling_strategy='auto',
    random_state=42,
    k_neighbors=5
)

X_balanced, y_balanced = smote.fit_resample(X, y)

print("BALANCEO COMPLETADO")

print("\nFEATURES BALANCEADAS:")
print(X_balanced.shape)

print("\nTARGET BALANCEADO:")
print(y_balanced.shape)

print("\nNUEVA DISTRIBUCIÓN:")
print(pd.Series(y_balanced).value_counts())

BALANCEO COMPLETADO

FEATURES BALANCEADAS:
(7644154, 21)

TARGET BALANCEADO:
(7644154,)

NUEVA DISTRIBUCIÓN:
traffic_label
1    3822077
0    3822077
Name: count, dtype: int64


In [7]:
# ============================================
# CELDA 7: CREAR DATASET BALANCEADO
# ============================================

balanced_df = pd.DataFrame(
    X_balanced,
    columns=X.columns
)

balanced_df["traffic_label"] = y_balanced

print(balanced_df.head())

print("\nSHAPE:")
print(balanced_df.shape)

   src_port  dst_port  protocol  duration_ms  bidirectional_packets  \
0 -2.740926 -0.353949 -1.153072    -0.159071              -0.006939   
1  0.833272 -0.209669  1.697311    -0.157960              -0.005903   
2 -2.740926 -0.353949 -1.153072    -0.159071              -0.006939   
3 -1.362775 -0.349924  1.697311    -0.158824              -0.006766   
4 -2.740926 -0.353949 -1.153072    -0.159071              -0.006939   

   bidirectional_bytes  src2dst_packets  dst2src_packets  src2dst_bytes  \
0            -0.016591        -0.004337        -0.006445      -0.010569   
1            -0.011866        -0.003011        -0.006445      -0.004041   
2            -0.016591        -0.004337        -0.006445      -0.010569   
3            -0.015823        -0.004337        -0.006132      -0.010413   
4            -0.016591        -0.004337        -0.006445      -0.010569   

   dst2src_bytes  ...  application_name  application_category  attack_type  \
0      -0.018081  ...               100     

In [8]:
# ============================================
# CELDA 8: EXPORTAR DATASET BALANCEADO
# ============================================

balanced_df.to_csv(
    OUTPUT_BALANCED,
    index=False
)

print("DATASET BALANCEADO EXPORTADO")

print(OUTPUT_BALANCED)

DATASET BALANCEADO EXPORTADO
e:\Proyectos VS code\ProcesamientoData\output_phase4\balanced_it_ot.csv


In [9]:
# ============================================
# CELDA 9: APLICAR PCA CORREGIDO
# ============================================

X_only = balanced_df.drop(columns=["traffic_label"])

y_only = balanced_df["traffic_label"]

pca = PCA(
    n_components=10,
    random_state=42
)

X_pca = pca.fit_transform(X_only)

print("PCA COMPLETADO")

print("\nSHAPE ORIGINAL:")
print(X_only.shape)

print("\nSHAPE REDUCIDO:")
print(X_pca.shape)

print("\nVARIANZA EXPLICADA TOTAL:")
print(np.sum(pca.explained_variance_ratio_))

print("\nVARIANZA POR COMPONENTE:")
print(pca.explained_variance_ratio_)

PCA COMPLETADO

SHAPE ORIGINAL:
(7644154, 21)

SHAPE REDUCIDO:
(7644154, 10)

VARIANZA EXPLICADA TOTAL:
0.9999999959440458

VARIANZA POR COMPONENTE:
[9.99996781e-01 2.74622397e-06 1.84798381e-07 1.60127850e-07
 8.24163357e-08 1.90725989e-08 1.01525394e-08 7.28499759e-09
 3.33009829e-09 1.31862172e-09]


In [10]:
# ============================================
# CELDA 10: CREAR DATASET REDUCIDO
# ============================================

pca_columns = []

for i in range(X_pca.shape[1]):
    pca_columns.append(f"PC_{i+1}")

reduced_df = pd.DataFrame(
    X_pca,
    columns=pca_columns
)

reduced_df["traffic_label"] = y_only.values

print(reduced_df.head())

print("\nSHAPE:")
print(reduced_df.shape)

print("\nCOLUMNAS:")
print(reduced_df.columns)

           PC_1       PC_2       PC_3       PC_4      PC_5      PC_6  \
0 -38367.102774 -20.507619  -4.939883  23.306892 -3.069444  4.116033   
1 -38366.101766  39.461572   6.956742  25.272827  0.347842  4.550396   
2 -38365.102776 -20.501382  -3.941216  23.294779 -3.091192  4.024311   
3 -38364.103582 -74.453146 -24.821463  22.593498 -6.335763  3.435141   
4 -38363.102780 -20.510990  -5.939638  23.315958 -3.051588  4.106636   

       PC_7      PC_8      PC_9     PC_10  traffic_label  
0 -0.872637 -0.888003 -0.341449  0.506612              1  
1  0.057176  1.712351 -0.417963 -0.294786              1  
2 -1.370249 -0.755539 -0.338369  0.503276              1  
3 -1.034161 -0.083315 -0.107417 -0.036456              1  
4 -0.868098 -0.885108 -0.343359  0.506974              1  

SHAPE:
(7644154, 11)

COLUMNAS:
Index(['PC_1', 'PC_2', 'PC_3', 'PC_4', 'PC_5', 'PC_6', 'PC_7', 'PC_8', 'PC_9',
       'PC_10', 'traffic_label'],
      dtype='str')


In [11]:
# ============================================
# CELDA 11: EXPORTAR DATASET REDUCIDO
# ============================================

reduced_df.to_csv(
    OUTPUT_PCA,
    index=False
)

print("DATASET PCA EXPORTADO")

print(OUTPUT_PCA)

DATASET PCA EXPORTADO
e:\Proyectos VS code\ProcesamientoData\output_phase4\reduced_it_ot.csv


In [12]:
# ============================================
# CELDA 12: GUARDAR MODELOS
# ============================================

joblib.dump(
    smote,
    os.path.join(OUTPUT_DIR, "smote_model.pkl")
)

joblib.dump(
    pca,
    os.path.join(OUTPUT_DIR, "pca_model.pkl")
)

print("MODELOS EXPORTADOS")

MODELOS EXPORTADOS


In [13]:
# ============================================
# CELDA 13: VALIDACIÓN FINAL
# ============================================

print("===================================")
print("FASE 4 COMPLETADA")
print("===================================")

print("\nDATASET BALANCEADO:")
print(OUTPUT_BALANCED)

print("\nDATASET REDUCIDO:")
print(OUTPUT_PCA)

print("\nTOTAL FEATURES PCA:")
print(X_pca.shape[1])

print("\nLISTO PARA:")
print("- TRAIN TEST SPLIT")
print("- XGBOOST")
print("- LIGHTGBM")
print("- RANDOM FOREST")
print("- DNN")
print("- STACKING ENSEMBLE")

FASE 4 COMPLETADA

DATASET BALANCEADO:
e:\Proyectos VS code\ProcesamientoData\output_phase4\balanced_it_ot.csv

DATASET REDUCIDO:
e:\Proyectos VS code\ProcesamientoData\output_phase4\reduced_it_ot.csv

TOTAL FEATURES PCA:
10

LISTO PARA:
- TRAIN TEST SPLIT
- XGBOOST
- LIGHTGBM
- RANDOM FOREST
- DNN
- STACKING ENSEMBLE
